**preproces**

In [35]:
import pandas as pd
import re
import string
!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# ==========================
# LOAD DATA
# ==========================
df = pd.read_csv("labeling data manual.csv")

# ==========================
# STEMMER
# ==========================
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# ==========================
# STOPWORDS AMAN (tidak hapus kata penting)
# ==========================
stopwords_custom = set([
    "yang","untuk","dan","di","ke","dari","ini","itu","juga","kami","kamu","saya","meirizka",
    "ada","dengan","pada","karena","agar","atau","jadi","tidak","iya","yg","nya","ya","tapi","powwl","ny","nih","lahh","zxz",
    "buat","dalam","para","akan","sudah","serta","sih","hehe","hmm","deh","mbak","mba","pool","tania",
    "mas","kak","caroline","devita","lena","dg","fifin","tp","dgn","loh","t","b","aja","imo","amoun",
    "saja","pun","itu","anda","agar","yakni","sebagai","maka","yaitu","silvia","ad","sy","san","kpd","the","untvlantai","lo",
    "bs","d","n","sll","aztuti","dkt", "tx", "u", "anak","aku","devita","best","dj","thania","fres","asa",
    "dll","yak","hahaha","ken","siip","kita","madam","diksh","kecil","h","yahh","x","mb","st","fo","silviaa","ka",
    "f","b","sm","tdk","nur","si","kap","m","yuuk","parpel","chelsi","dg","sgt,","tpi","se","occ","daan",
    "an","andin","traine","trainee","s","mau","kes","asa","jd","ku","dik","welcome","dr","thania"
])

# KATA PENTING DOMAIN HOTEL — JANGAN DIHAPUS
stopword_protect = {"hotel","kamar","lokasi","pelayanan","fasilitas","staf","sarapan",
                    "bersih","nyaman","air","ac","wifi","parkir","resepsionis",
                    "checkin","checkout","kasur","bantal"}
stopwords = stopwords_custom - stopword_protect

# ==========================
# NORMALISASI SLANG (fix conflict & duplicates)
# ==========================
slang = {
    "bgtt":"banget","bgt":"banget","bangett":"banget","byk":"banyak","inn":"in","bnget":"banget","kren":"keren","suuperr":"bagus","servicenha":"servis",
    "gpp":"gapapa","ga":"tidak","gk":"tidak","nggak":"tidak","ngga":"tidak","kram":"kran","dkt":"dekat","cek":"check","pesen":"pesan",
    "makasih":"terima kasih","mksh":"terima kasih","thx":"terima kasih","jl":"jalan","dtg":"datang","baguss":"bagus","thank you":"terimakasih",
    "oke":"baik","ok":"baik","sip":"baik","bberapa":"beberapa","asa":"aja","lg":"lagi","msk":"masuk","tv":"televisi","clear":"bersih","tsb":"tersebut","overalla":"over all",
    "rekomen":"rekomendasi","recommended":"rekomendasi","sangatt":"sangat","maknyus":"enak","topp":"bagus","thank u":"terimaksih","tengkyu":"terimakasih",
    "jg":"juga","jd":"jadi","dpt":"dapat","dapet":"dapat","apalgii":"apalagi","restorannlt":"restoran","desertnya":"desert","trmmksh":"terimakasih",
    "km":"kamu","tmn":"teman","tmpt":"tempat","tks":"terimakasih","berasa":"rasa","ksni":"kesini","sus":"aneh","baguus":"bagus","besty":"best","trimakasih":"terimakasih",
    "udh":"sudah","sdh":"sudah","blm":"belum","hr":"hari","wlopun":"walaupun","happy":"senang","bussines trip":"perjalanan bisnis","ga":"tidak",
    "kl":"kalau","klo":"kalau","kalo":"kalau","tksh":"terimakasih","cantikk":"cantik","nggk":"tidak","menservice":"servis","overall":"semua","sat set":"cepat",
    "krn":"karena","trm":"terima","trs":"terus","bgs":"bagus","breakfastnya":"breakfast","enakk":"enak","thank":"terimakasih","lt":"lantai",
    "bbrp":"beberapa","kmr":"kamar","dgn":"dengan","lbh":"lebih","enakk":"enak","baguss":"bagus","trus":"terus","gak":"tidak","bnyk":"banyak","amburadul":"berantakan",
    "mantappp":"mantap","mantapppp":"mantap","mantappppp":"mantap","sukakk":"suka","lonsay":"lontong sayur","too bad":"buruk","deal":"setuju","tgl":"tanggal","agam":"agak","kg":"juga",
    "parahh":"parah","pdhal":"padahal","tpat":"tempat","lg":"lagi","yogjakarta":"yogyakarta","pelayannann":"pelayanan","recomend":"recomended",
    "thank u":"terima kasih","utk":"untuk","org":"orang","pengkap":"lengkap","rs":"rumah sakit","breakfast":"sarapan","pdhl":"padahal","bukber":"buka bersama","hotell":"hotel"
}

def normalize_slang(word):
    return slang.get(word, word)

# ==========================
# FULL PREPROCESSING FUNCTION
# ==========================
def preprocess_full(text):
    if pd.isna(text):
        return ""

    # 1. Casefolding
    t = text.lower()

    # 2. Clean HTML tags
    t = re.sub(r"<.*?>", " ", t)

    # 3. Remove URL
    t = re.sub(r"http\S+|www\.\S+", "", t)

    # 4. Remove @mention
    t = re.sub(r"@\w+", "", t)

    # 5. Remove emoji
    t = t.encode('ascii', 'ignore').decode('ascii')

    # 6. Remove punctuation
    t = t.translate(str.maketrans("", "", string.punctuation))

    # 7. Remove numbers
    t = re.sub(r"\d+", "", t)

    # 8. Tokenize words
    tokens = t.split()

    # 9. Normalisasi slang
    tokens = [normalize_slang(w) for w in tokens]

    # 10. Remove repeated characters
    tokens = [re.sub(r"(.)\1{2,}", r"\1", w) for w in tokens]

    # 11. Stopword removal
    tokens = [w for w in tokens if w not in stopwords]

    # 12. Stemming Sastrawi
    tokens = [stemmer.stem(w) for w in tokens]

    # 13. Remove extra whitespace
    return " ".join(tokens)

# ==========================
# APPLY TO DATAFRAME
# ==========================
df["processed"] = df["Casefold"].astype(str).apply(preprocess_full)

df.to_csv("reviews_processed_full.csv", index=False)

print("Preprocessing FINAL selesai! File: reviews_processed_full.csv")
df.head()

Preprocessing FINAL selesai! File: reviews_processed_full.csv


,Label,Casefold,processed
0,positif,pelayanannya ramah bgtt fasilitas kamar juga o...,layan ramah banget fasilitas kamar baik rekome...
1,negatif,baru kali ini sangat kecewa dengan pelayanan h...,baru kali sangat kecewa layan hotel mercure ko...
2,positif,liburan menyenangkan di jogja stay di hotel yg...,libur senang jogja stay hotel luar biasa
3,positif,lokasinya juga mantap dekat dengan bandara dan...,lokasi mantap dekat bandara tempat tempat wisa...
4,positif,saya merekomendasikan anda untuk menginap di h...,rekomendasi inap hotel layan bagus ramah makan...


In [36]:
# file: preprocessing.py
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# Load dataset (sudah berlabel dan tidak)
dataset = pd.read_csv('/content/reviews_processed_full.csv')

# Pisahkan data berlabel
manual = dataset[~dataset['Label'].isna()]

# Tokenizer
max_words = 10000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(manual['Casefold'])

def preprocess_text(df):
    sequences = tokenizer.texts_to_sequences(df['Casefold'])
    padded = pad_sequences(sequences, maxlen=max_len)
    return padded

# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

**train**

In [37]:
# file: train_initial_model.py
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences # Import pad_sequences needed for preprocess_text

# Load dataset (using the correct path)
dataset = pd.read_csv('/content/reviews_processed_full.csv')
manual = dataset[~dataset['Label'].isna()].copy() # Corrected column name and added .copy() to avoid SettingWithCopyWarning

# Encode label
le = LabelEncoder()
manual['label_encoded'] = le.fit_transform(manual['Label']) # Corrected column name

# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Define preprocess_text function here (after tokenizer is loaded)
max_len = 100 # Define max_len for preprocess_text
def preprocess_text(df):
    sequences = tokenizer.texts_to_sequences(df['Casefold'])
    padded = pad_sequences(sequences, maxlen=max_len)
    return padded

# Preprocess
X_labeled = preprocess_text(manual)
y_labeled = manual['label_encoded'].values

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X_labeled, y_labeled, test_size=0.2, random_state=42)

# Build model
max_words = 10000
# max_len = 100 # Already defined above for preprocess_text
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128)) # Removed deprecated input_length argument
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(len(manual['Label'].unique()), activation='softmax')) # Corrected column name

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=10, batch_size=32, callbacks=[early_stop])

# Save model
model.save('model_initial.h5') # Consider changing to 'model_initial.keras' for the latest format
print("Initial model trained and saved as model_initial.h5")

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 177ms/step - accuracy: 0.3683 - loss: 1.0916 - val_accuracy: 0.5600 - val_loss: 1.0165
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.5386 - loss: 1.0443 - val_accuracy: 0.4400 - val_loss: 0.9309
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 0.5057 - loss: 0.9755 - val_accuracy: 0.7400 - val_loss: 0.8519
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 201ms/step - accuracy: 0.6040 - loss: 0.8872 - val_accuracy: 0.7400 - val_loss: 0.7626
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 144ms/step - accuracy: 0.6600 - loss: 0.7624 - val_accuracy: 0.7600 - val_loss: 0.6502
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.6817 - loss: 0.6760 - val_accuracy: 0.8000 - val_loss: 0.5430
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.7630 - loss: 0.5437 - val_accuracy: 0.8600 - val_loss: 0.4636
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.7553 - loss: 0.4925 - val_accuracy: 0.9000 - val_loss:

Initial model trained and saved as model_initial.h5


**self train**

In [38]:
# file: self_training.py
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder # Added for consistent encoding

# Load model awal
model = load_model('model_initial.h5')

# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Define max_len (must be consistent with tokenizer training)
max_len = 100

# Define preprocess_text function here
def preprocess_text(df):
    sequences = tokenizer.texts_to_sequences(df['Casefold'])
    padded = pad_sequences(sequences, maxlen=max_len)
    return padded

# Load dataset (using the correct path)
dataset = pd.read_csv('/content/reviews_processed_full.csv') # Corrected file path

# Separate data
manual = dataset[~dataset['Label'].isna()].copy() # Corrected column name and added .copy()
unlabeled = dataset[dataset['Label'].isna()].copy() # Corrected column name and added .copy()

# Fill NaN values in 'Casefold' column with empty strings before preprocessing
unlabeled['Casefold'] = unlabeled['Casefold'].fillna('')

# Encode manual labels for consistency with pseudo-labels
le = LabelEncoder()
manual['label_encoded'] = le.fit_transform(manual['Label']) # Re-encode manual labels

# Preprocess unlabeled data
X_unlabeled = preprocess_text(unlabeled)

# Prediksi pseudo-label
pred_probs = model.predict(X_unlabeled)
pred_labels = np.argmax(pred_probs, axis=1)
pred_confidence = np.max(pred_probs, axis=1)

# Threshold confidence
threshold = 0.9
high_conf_idx = np.where(pred_confidence >= threshold)[0]

pseudo_data = unlabeled.iloc[high_conf_idx].copy()
pseudo_data['label_encoded'] = pred_labels[high_conf_idx]

# Gabungkan manual + pseudo-labeled
train_self = pd.concat([manual, pseudo_data], ignore_index=True)

# Simpan ke CSV
train_self.to_csv('data_self_train.csv', index=False)
print("Self-training done. data_self_train.csv siap digunakan.")

31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
Self-training done. data_self_train.csv siap digunakan.


In [39]:
# file: retrain_self_train_model.py
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import pickle
# Removed: from preprocessing import preprocess_text

# Load self-train dataset
train_self = pd.read_csv('data_self_train.csv')

# Fill NaN values in 'Casefold' column with empty strings
train_self['Casefold'] = train_self['Casefold'].fillna('')

# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Define max_len (must be consistent with tokenizer training)
max_len = 100

# Define preprocess_text function here, as it's no longer imported
def preprocess_text(df):
    sequences = tokenizer.texts_to_sequences(df['Casefold']) # Assuming 'Casefold' is the text column
    padded = pad_sequences(sequences, maxlen=max_len)
    return padded

# Preprocess
X_self = preprocess_text(train_self)
y_self = train_self['label_encoded'].values

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X_self, y_self, test_size=0.2, random_state=42)

# Build model
max_words = 10000
# max_len = 100 # Already defined above
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128)) # Removed deprecated input_length argument
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(len(train_self['label_encoded'].unique()), activation='softmax')) # Corrected column name to 'label_encoded'

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=10, batch_size=32, callbacks=[early_stop])

# Save final model
model.save('model_self_train.h5')
print("Self-training model retrained and saved as model_self_train.h5")

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 147ms/step - accuracy: 0.6253 - loss: 0.9033 - val_accuracy: 0.7664 - val_loss: 0.5537
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.8042 - loss: 0.4556 - val_accuracy: 0.8613 - val_loss: 0.4111
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - accuracy: 0.8521 - loss: 0.4049 - val_accuracy: 0.8759 - val_loss: 0.3463
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 173ms/step - accuracy: 0.8973 - loss: 0.2463 - val_accuracy: 0.8978 - val_loss: 0.2503
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 142ms/step - accuracy: 0.9217 - loss: 0.1878 - val_accuracy: 0.9051 - val_loss: 0.2372
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step - accuracy: 0.9051 - loss: 0.1611 - val_accuracy: 0.9051 - val_loss: 0.2247
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.9179 - loss: 0.1529 - val_accuracy: 0.9124 - val_loss: 0.2477
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - accuracy: 0.9505 - loss: 0.1071 - val_accuracy: 0.

Self-training model retrained and saved as model_self_train.h5
